[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_5_aruco/15_aruco_robotics_app.ipynb)

# Notebook 15 — ArUco Robotics Application

**Part 5: ArUco Markers** | Estimated time: 60 min

---

This is the **capstone of Part 5**. We build a complete robotics application:

> A mobile robot approaches a docking station equipped with an ArUco marker.  
> The camera estimates the marker pose, and from that we compute  
> **lateral error**, **heading error**, and **remaining distance** —  
> everything the robot needs to align and dock.

```
Full pipeline (all 4 ArUco notebooks):

  [NB 11] Theory → [NB 12] Generate → [NB 13] Detect → [NB 14] Pose → [NB 15] Robotics App
                                                                               ← YOU ARE HERE
```

## Recommended Watch

> Watch the full pipeline demo **before** opening this robotics application notebook — it's the end-to-end reference for everything built here.

| Title | Link | Duration |
|---|---|---|
| **ArUco Marker Pose Estimation and Detection in Real-Time using OpenCV Python** | [▶ Watch](https://www.youtube.com/watch?v=bS00Vs09Upw) | ~25 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python matplotlib numpy -q
    print('Running in Google Colab — packages installed')
else:
    print('Running locally — make sure your venv is activated')

## Learning Objectives

By the end of this notebook you will be able to:

- Translate ArUco pose estimates into robot-meaningful alignment commands
- Identify the correct station from marker ID
- Compute lateral error, heading error, and distance-to-dock
- Build a decision loop that guides a robot toward a station
- Understand coordinate frame transforms between camera and robot body

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from dataclasses import dataclass
from typing import Optional, Dict

print(f'OpenCV version: {cv2.__version__}')

def show_bgr(img, title='', figsize=(10, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 1. The Robotics Scenario

```
Factory Floor Layout:

  ┌────────────────────────────────────────────────────┐
  │                                                    │
  │  [Charging Station]   [Conveyor A]   [Conveyor B]  │
  │    ArUco ID=0          ArUco ID=1     ArUco ID=2   │
  │         ■                   ■              ■       │
  │                                                    │
  │                                                    │
  │            🤖 Mobile Robot (camera on front)       │
  └────────────────────────────────────────────────────┘
```

**The docking problem:**
1. Robot receives a command: "Go to Conveyor A"
2. Robot navigates roughly to station area (SLAM/odometry handles this)
3. Robot sees the ArUco marker with its front camera
4. **Our code** computes: how far left/right am I? How far away? Am I facing it straight?
5. Robot makes small corrective movements until aligned
6. Robot executes the docking action

In [ ]:
# ============================================================
# Station Registry — maps ArUco ID to station properties
# ============================================================
@dataclass
class Station:
    id:             int    # ArUco marker ID
    name:           str    # Human-readable name
    marker_length:  float  # Physical marker size in meters
    dock_distance:  float  # Target approach distance in meters
    dock_tolerance: float  # Acceptable position error in meters

STATIONS: Dict[int, Station] = {
    0: Station(id=0, name='Charging Station', marker_length=0.15, dock_distance=0.30, dock_tolerance=0.02),
    1: Station(id=1, name='Conveyor A',       marker_length=0.15, dock_distance=0.25, dock_tolerance=0.015),
    2: Station(id=2, name='Conveyor B',       marker_length=0.15, dock_distance=0.25, dock_tolerance=0.015),
}

print('Station registry:')
for sid, s in STATIONS.items():
    print(f'  ID={sid}: {s.name:20s} | marker={s.marker_length*100:.0f}cm | dock_dist={s.dock_distance*100:.0f}cm')

## 2. Coordinate Frame Transforms

The pose estimate `(rvec, tvec)` gives us the marker position in the **camera frame**.  
The robot controller needs positions in the **robot body frame**.

```
Camera coordinate frame:          Robot body frame:

    Z (depth, into scene)              X (forward)
    │                                  │
    │                                  │
    └── X (right)                      └── Y (left)
   /                                  /
  Y (down)                           Z (up)
```

For a camera mounted facing forward at the robot's "head":

| Camera tvec | Robot interpretation |
|---|---|
| `tvec[2]` (Z, depth) | Forward distance to station |
| `tvec[0]` (X, right) | Lateral offset (robot must move left/right) |
| `tvec[1]` (Y, down) | Vertical offset (less important for ground robots) |

In [ ]:
@dataclass
class AlignmentError:
    """Computed alignment errors for one detected station."""
    station:        Station
    distance_m:     float   # Z-depth: distance from robot to marker
    lateral_m:      float   # X-offset: left/right misalignment
    vertical_m:     float   # Y-offset: up/down misalignment (usually ignored)
    heading_deg:    float   # Yaw angle: how much to rotate to face marker
    is_aligned:     bool    # True if within tolerance on all axes
    is_at_distance: bool    # True if close enough to dock

def compute_alignment(tvec: np.ndarray, rvec: np.ndarray,
                      station: Station) -> AlignmentError:
    """
    Compute alignment errors from ArUco pose to docking target.
    
    Args:
        tvec:    (3,) translation vector in meters (camera frame)
        rvec:    (3,) Rodrigues rotation vector
        station: target station with dock parameters
    
    Returns:
        AlignmentError with all computed metrics
    """
    tx, ty, tz = tvec.flatten()
    
    # Distance from camera to marker (forward)
    distance_m = float(tz)     # depth along Z axis
    lateral_m  = float(tx)     # left/right offset
    vertical_m = float(ty)     # up/down offset
    
    # Heading error: extract yaw angle from rotation
    R, _ = cv2.Rodrigues(rvec)
    # Yaw is rotation around Y axis (camera convention)
    # atan2(R[0,2], R[2,2]) gives approximate heading
    heading_rad = np.arctan2(R[0, 2], R[2, 2])
    heading_deg = float(np.degrees(heading_rad))
    
    # Tolerances
    tol = station.dock_tolerance
    is_aligned = (
        abs(lateral_m) < tol and
        abs(heading_deg) < 5.0  # within 5 degrees heading
    )
    is_at_distance = abs(distance_m - station.dock_distance) < tol
    
    return AlignmentError(
        station        = station,
        distance_m     = distance_m,
        lateral_m      = lateral_m,
        vertical_m     = vertical_m,
        heading_deg    = heading_deg,
        is_aligned     = is_aligned,
        is_at_distance = is_at_distance,
    )

print('AlignmentError dataclass and compute_alignment() ready.')

## 3. Robot Command Generation

From the alignment errors, we generate simple proportional commands.  
In a real system these would go to the motor controller.

```
Proportional control (simplified):

  forward_speed  = Kp_dist  × (distance - dock_distance)
  lateral_speed  = Kp_lat   × lateral_error
  rotation_speed = Kp_head  × heading_error
```

In [ ]:
@dataclass
class RobotCommand:
    forward:  float  # m/s — positive = forward, negative = backward
    lateral:  float  # m/s — positive = right, negative = left
    rotation: float  # deg/s — positive = clockwise, negative = counter-clockwise
    action:   str    # human-readable state

# Proportional control gains
KP_DIST  = 0.5   # m/s per meter of distance error
KP_LAT   = 1.0   # m/s per meter of lateral error
KP_HEAD  = 2.0   # deg/s per degree of heading error
MAX_FWD  = 0.3   # m/s max forward speed
MAX_LAT  = 0.15  # m/s max lateral speed
MAX_ROT  = 30.0  # deg/s max rotation speed

def generate_command(err: AlignmentError) -> RobotCommand:
    """
    Generate robot velocity command from alignment error.
    Priority: 1. Correct heading  2. Correct lateral  3. Approach
    """
    if err.is_aligned and err.is_at_distance:
        return RobotCommand(0.0, 0.0, 0.0, 'DOCKED')
    
    # 1. Heading correction (rotate to face marker)
    rot = np.clip(-KP_HEAD * err.heading_deg, -MAX_ROT, MAX_ROT)
    
    # 2. Lateral correction (strafe left/right)
    lat = np.clip(-KP_LAT * err.lateral_m, -MAX_LAT, MAX_LAT)
    
    # 3. Forward approach
    dist_error = err.distance_m - err.station.dock_distance
    fwd = np.clip(KP_DIST * dist_error, -MAX_FWD, MAX_FWD)
    
    # Determine action label
    if err.is_at_distance and err.is_aligned:
        action = 'DOCKED'
    elif err.is_aligned:
        action = 'APPROACH' if dist_error > 0 else 'BACK_UP'
    else:
        action = 'ALIGNING'
    
    return RobotCommand(forward=float(fwd), lateral=float(lat),
                        rotation=float(rot), action=action)

print('generate_command() ready.')

## 4. Full Pipeline Setup

The previous sections built the individual pieces: coordinate frame transforms (Section 2) and proportional command generation (Section 3). This section assembles them into the complete perception-to-command pipeline — a single function that takes a raw camera frame and returns robot motion commands.

In [ ]:
# --- Camera calibration ---
CALIB_PATH = '../assets/calibration/camera_calibration.npz'
if os.path.exists(CALIB_PATH):
    data = np.load(CALIB_PATH)
    K    = data['camera_matrix']
    dist = data['dist_coeffs']
    print(f'Loaded calibration from {CALIB_PATH}')
else:
    K = np.array([[600, 0, 320],[0, 600, 240],[0, 0, 1]], dtype=np.float64)
    dist = np.zeros((1, 5), dtype=np.float64)
    print('Using default K (run notebook 07 for real calibration)')

# --- ArUco detector ---
def get_aruco_dict(dict_id):
    try:
        return cv2.aruco.getPredefinedDictionary(dict_id)
    except AttributeError:
        return cv2.aruco.Dictionary_get(dict_id)

def generate_marker(aruco_dict, marker_id, size_px):
    try:
        return cv2.aruco.generateImageMarker(aruco_dict, marker_id, size_px)
    except AttributeError:
        img = np.zeros((size_px, size_px, 1), dtype='uint8')
        cv2.aruco.drawMarker(aruco_dict, marker_id, size_px, img, 1)
        return img.squeeze()

aruco_dict_obj = get_aruco_dict(cv2.aruco.DICT_4X4_50)

try:
    params = cv2.aruco.DetectorParameters()
    params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
    detector = cv2.aruco.ArucoDetector(aruco_dict_obj, params)
    USE_NEW_API = True
except AttributeError:
    params = cv2.aruco.DetectorParameters_create()
    USE_NEW_API = False

def detect_and_pose(frame: np.ndarray, target_station: Station):
    """
    Full pipeline: detect → pose estimate → return results.
    Only returns results if the target station's marker ID is found.
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    if USE_NEW_API:
        corners, ids, rejected = detector.detectMarkers(gray)
    else:
        corners, ids, rejected = cv2.aruco.detectMarkers(
            gray, aruco_dict_obj, parameters=params)
    
    if ids is None:
        return None, None, None, corners, ids, rejected
    
    flat_ids = ids.flatten()
    
    # Find our target station's marker
    for i, mid in enumerate(flat_ids):
        if mid == target_station.id:
            # Estimate pose for this specific marker
            corner_single = [corners[i]]
            rvecs, tvecs, _ = cv2.aruco.estimatePoseSingleMarkers(
                corner_single,
                target_station.marker_length,
                K, dist
            )
            return rvecs[0], tvecs[0], corners[i], corners, ids, rejected
    
    return None, None, None, corners, ids, rejected

print('Full pipeline ready.')
print(f'ArUco API: {"new" if USE_NEW_API else "old"}')

## 5. HUD Overlay — Robot's View

The robot operator (or the robot's own monitoring system) needs to see what the perception system is seeing. Raw camera feed isn't enough — you need alignment data overlaid directly on the image so the current state is readable at a glance.

The HUD (Heads-Up Display) draws lateral offset, approach distance, heading error, and docking state directly onto the frame. This is the same pattern used in autonomous vehicle dashboards and industrial robot interfaces.

In [ ]:
def draw_docking_hud(frame: np.ndarray,
                     target_station: Station,
                     rvec, tvec,
                     target_corner,
                     all_corners, all_ids,
                     err: Optional['AlignmentError'],
                     cmd: Optional['RobotCommand']) -> np.ndarray:
    """
    Draw a HUD overlay showing station ID, alignment errors, and command.
    """
    output = frame.copy()
    h, w = output.shape[:2]
    
    # Draw all detected markers (dim)
    if all_ids is not None:
        cv2.aruco.drawDetectedMarkers(output, all_corners, all_ids)
    
    if rvec is not None and tvec is not None and target_corner is not None:
        # Draw axes for target marker
        cv2.drawFrameAxes(output, K, dist, rvec, tvec,
                          target_station.marker_length * 0.6)
        
        # Highlight target marker outline in bright yellow
        pts = target_corner.reshape(4, 2).astype(int)
        for j in range(4):
            cv2.line(output, tuple(pts[j]), tuple(pts[(j+1)%4]),
                     (0, 255, 255), 3)
        
        # Draw center crosshair
        cx = int(pts[:, 0].mean())
        cy = int(pts[:, 1].mean())
        cv2.drawMarker(output, (cx, cy), (0, 255, 255),
                       cv2.MARKER_CROSS, 20, 2)
    
    # HUD background panel
    panel_h = 160
    cv2.rectangle(output, (0, 0), (w, panel_h), (0, 0, 0), -1)
    cv2.rectangle(output, (0, 0), (w, panel_h), (80, 80, 80), 1)
    
    # Station info
    cv2.putText(output, f'TARGET: {target_station.name} (ID={target_station.id})',
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 200, 0), 2)
    
    if err is None:
        cv2.putText(output, 'SEARCHING... marker not visible',
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 100, 255), 2)
    else:
        # Alignment metrics
        lat_ok  = abs(err.lateral_m)  < target_station.dock_tolerance
        dist_ok = err.is_at_distance
        head_ok = abs(err.heading_deg) < 5.0
        
        def ok_color(flag): return (0, 220, 0) if flag else (0, 100, 255)
        
        cv2.putText(output,
                    f'Dist:    {err.distance_m*100:+.1f} cm  (target: {target_station.dock_distance*100:.0f} cm)',
                    (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.6, ok_color(dist_ok), 2)
        cv2.putText(output,
                    f'Lateral: {err.lateral_m*100:+.1f} cm  (target: 0 cm)',
                    (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, ok_color(lat_ok), 2)
        cv2.putText(output,
                    f'Heading: {err.heading_deg:+.1f} deg (target: 0 deg)',
                    (10, 118), cv2.FONT_HERSHEY_SIMPLEX, 0.6, ok_color(head_ok), 2)
        
        if cmd is not None:
            action_color = (0, 220, 0) if cmd.action == 'DOCKED' else (255, 200, 0)
            cv2.putText(output,
                        f'CMD: {cmd.action} | fwd={cmd.forward:+.2f}m/s lat={cmd.lateral:+.2f}m/s rot={cmd.rotation:+.1f}dps',
                        (10, 148), cv2.FONT_HERSHEY_SIMPLEX, 0.55, action_color, 2)
    
    # Camera crosshair center
    cv2.line(output, (w//2-15, h//2), (w//2+15, h//2), (100,100,100), 1)
    cv2.line(output, (w//2, h//2-15), (w//2, h//2+15), (100,100,100), 1)
    
    return output

print('HUD overlay function ready.')

## 6. Simulating Different Approach Scenarios

We simulate three robot positions:
- **Far away, centered** — approaching but not aligned yet
- **Close, misaligned** — needs lateral correction
- **Perfectly aligned** — ready to dock

In [ ]:
def simulate_robot_view(station_id: int, marker_px_size: int,
                        offset_x: int = 0, frame_size: int = 640):
    """
    Simulate what a robot camera would see at a given distance and lateral offset.
    
    marker_px_size: larger = closer to station
    offset_x: horizontal pixel offset of marker center from image center
    """
    frame = np.ones((frame_size, frame_size, 3), dtype=np.uint8) * 180
    # Add a simple factory wall texture
    for row in range(0, frame_size, 60):
        cv2.line(frame, (0, row), (frame_size, row), (160,160,160), 1)
    for col in range(0, frame_size, 60):
        cv2.line(frame, (col, 0), (col, frame_size), (160,160,160), 1)
    
    # Generate marker
    aruco_d = get_aruco_dict(cv2.aruco.DICT_4X4_50)
    m = generate_marker(aruco_d, station_id, marker_px_size)
    m_bgr = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
    
    # Place marker center at (frame_center + offset_x, frame_center_y)
    cx = frame_size // 2 + offset_x
    cy = frame_size // 2 - 30  # Slightly above center (mounted on wall)
    x0 = cx - marker_px_size // 2
    y0 = cy - marker_px_size // 2
    x1 = x0 + marker_px_size
    y1 = y0 + marker_px_size
    
    # Clip
    sx0 = max(0, x0); sy0 = max(0, y0)
    sx1 = min(frame_size, x1); sy1 = min(frame_size, y1)
    mx0 = sx0 - x0; my0 = sy0 - y0
    frame[sy0:sy1, sx0:sx1] = m_bgr[my0:my0+(sy1-sy0), mx0:mx0+(sx1-sx0)]
    
    return frame

# Three approach scenarios
TARGET_STATION = STATIONS[1]  # Conveyor A

scenarios = [
    {'name': 'Far away, centered',    'px': 80,  'offset': 0},
    {'name': 'Close, lateral offset', 'px': 200, 'offset': 120},
    {'name': 'Aligned, correct dist', 'px': 155, 'offset': 5},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

for ax, scenario in zip(axes, scenarios):
    frame = simulate_robot_view(TARGET_STATION.id,
                                 scenario['px'],
                                 scenario['offset'])
    rvec, tvec, target_corner, all_c, all_i, all_r = detect_and_pose(frame, TARGET_STATION)
    
    err = compute_alignment(tvec, rvec, TARGET_STATION) if rvec is not None else None
    cmd = generate_command(err) if err is not None else None
    
    hud = draw_docking_hud(frame, TARGET_STATION, rvec, tvec,
                            target_corner, all_c, all_i, err, cmd)
    
    ax.imshow(cv2.cvtColor(hud, cv2.COLOR_BGR2RGB))
    ax.set_title(scenario['name'], fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle('Robot docking approach — three phases', fontsize=14)
plt.tight_layout()
plt.show()

## 7. The Full Decision Loop

In production, this runs every frame from the camera:

In [ ]:
def docking_pipeline_step(frame: np.ndarray, target_station: Station):
    """
    One iteration of the docking control loop.
    
    Returns:
        hud_frame:  annotated frame for display
        command:    RobotCommand or None if marker not found
        error:      AlignmentError or None
    """
    # Step 1: Detect + Pose estimate
    rvec, tvec, target_corner, all_c, all_i, all_r = detect_and_pose(frame, target_station)
    
    # Step 2: Compute alignment
    err = compute_alignment(tvec, rvec, target_station) if rvec is not None else None
    
    # Step 3: Generate command
    cmd = generate_command(err) if err is not None else None
    
    # Step 4: Draw HUD
    hud = draw_docking_hud(
        frame, target_station, rvec, tvec,
        target_corner, all_c, all_i, err, cmd
    )
    
    return hud, cmd, err

# Test the full loop on our scenarios
print('=== Decision Loop Results ===')
print(f'Target: {TARGET_STATION.name} (ID={TARGET_STATION.id})')
print()

for scenario in scenarios:
    frame = simulate_robot_view(TARGET_STATION.id, scenario['px'], scenario['offset'])
    hud, cmd, err = docking_pipeline_step(frame, TARGET_STATION)
    
    print(f"Scenario: {scenario['name']}")
    if err is None:
        print('  → SEARCH (no marker detected)')
    else:
        print(f'  Dist:    {err.distance_m*100:.1f}cm (target {TARGET_STATION.dock_distance*100:.0f}cm)')
        print(f'  Lateral: {err.lateral_m*100:+.1f}cm')
        print(f'  Heading: {err.heading_deg:+.1f}°')
        print(f'  → CMD: {cmd.action} | fwd={cmd.forward:+.2f} lat={cmd.lateral:+.2f} rot={cmd.rotation:+.1f}')
    print()

## 8. Real-Time Docking Script (for local use)

Everything in this notebook runs on synthetic frames. The real-time version below replaces the synthetic frames with `cv2.VideoCapture` — connect a real camera and point it at a printed ArUco marker at the configured ID and size, and this script becomes a functional docking guidance system.

Save it as `aruco_docking.py` and run locally. You need a webcam and a printed marker (use NB 12 to generate it at the correct size).

In [ ]:
# The real-time docking script lives in scripts/robotics/robot_station_docking.py
# Run it from the repo root in a terminal:
#
#   python scripts/robotics/robot_station_docking.py
#
# Optional arguments:
#   --calib   PATH    calibration .npz from NB07 (default: auto-detect)
#   --target  INT     initial target station ID (default: 1)
#   --camera  INT     webcam index (default: 0)
#
# Controls while running:
#   Q / Esc     → quit
#   0 / 1 / 2   → switch target station
#
# Edit STATIONS dict at the top of the script to match your physical setup:
#   marker size, dock distance, tolerance.
#
# Output: forward/lateral/rotation commands + docking state HUD overlay.

print("See scripts/robotics/robot_station_docking.py — run from repo root in a terminal.")

## Recap

| Concept | Key Point |
|---|---|
| Station registry | Map marker ID → physical properties (size, dock distance) |
| `tvec[2]` = depth | The most important value — distance to dock |
| `tvec[0]` = lateral | Left/right misalignment — drives strafe command |
| Heading from rvec | Extract yaw angle via Rodrigues → rotation matrix |
| P-controller | Simple proportional gain — sufficient for slow docking |
| Decision states | SEARCH → ALIGNING → APPROACH → DOCKED |

**Part 5 Complete!** You can now:
- Generate ArUco markers for stations (NB 12)
- Detect them in real-time (NB 13)
- Estimate their 6D pose (NB 14)
- Use pose to guide a robot to dock (NB 15)

**Next:** Part 6 — Stereo Vision for depth estimation without markers.

---
## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Add a 4th station
# ============================================================
# Add a new station: {'id': 3, 'name': 'Quality Check', marker_length=0.12,
#                     dock_distance=0.20, dock_tolerance=0.01}
# Simulate approaching it and print the alignment report.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# qc_station = Station(id=3, name='Quality Check',
#                      marker_length=0.12, dock_distance=0.20, dock_tolerance=0.01)
# frame = simulate_robot_view(3, 160, 30)
# hud, cmd, err = docking_pipeline_step(frame, qc_station)
# show_bgr(hud, 'Quality Check station approach')
# if err:
#     print(f'Dist: {err.distance_m*100:.1f}cm, Lateral: {err.lateral_m*100:+.1f}cm')
#     print(f'CMD: {cmd.action}')

In [ ]:
# ============================================================
# EXERCISE 2: Multi-station scene
# ============================================================
# Create a scene with markers 0, 1, 2 all visible.
# Run the pipeline for each station.
# Print which one is currently closest.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# frame = np.ones((640, 900, 3), dtype=np.uint8) * 180
# aruco_d = get_aruco_dict(cv2.aruco.DICT_4X4_50)
# for i, (sid, px, x0) in enumerate([(0,120,50),(1,100,330),(2,90,610)]):
#     m = generate_marker(aruco_d, sid, px)
#     m_bgr = cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
#     frame[200:200+px, x0:x0+px] = m_bgr
# 
# closest = None; closest_dist = float('inf')
# for sid, station in STATIONS.items():
#     _, cmd, err = docking_pipeline_step(frame, station)
#     if err:
#         print(f'Station {sid} ({station.name}): Z={err.distance_m*100:.1f}cm')
#         if err.distance_m < closest_dist:
#             closest_dist = err.distance_m; closest = station.name
# print(f'Closest: {closest}')

In [ ]:
# ============================================================
# EXERCISE 3: Approach simulation plot
# ============================================================
# Simulate approaching station 1 from far (px=60) to close (px=220)
# in 10 steps. For each step, record distance and lateral error.
# Plot both as a function of step number.
# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# station = STATIONS[1]
# sizes = np.linspace(60, 220, 10, dtype=int)
# dists, laterals = [], []
# 
# for px in sizes:
#     frame = simulate_robot_view(1, int(px), offset_x=40)
#     _, cmd, err = docking_pipeline_step(frame, station)
#     if err:
#         dists.append(err.distance_m * 100)
#         laterals.append(err.lateral_m * 100)
#     else:
#         dists.append(None); laterals.append(None)
# 
# fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
# steps = range(len(sizes))
# ax1.plot(steps, dists, 'b-o'); ax1.axhline(station.dock_distance*100, color='r', linestyle='--', label='target')
# ax1.set_ylabel('Distance (cm)'); ax1.legend(); ax1.set_title('Approach simulation')
# ax2.plot(steps, laterals, 'g-o'); ax2.axhline(0, color='r', linestyle='--')
# ax2.set_ylabel('Lateral (cm)'); ax2.set_xlabel('Step')
# plt.tight_layout(); plt.show()